# Oracle DML：Student-t 与 Lognormal（$p=5$）

本 notebook 使用真实 nuisance functions，检验可行 DML 的低覆盖率是否主要来自第一阶段机器学习误差。保留原实验的 $p=5$、$N=1000$、$\theta_0=1$、两种 DGP、误差参数网格、随机种子和 1000 次 Monte Carlo 重复。

模型为
$$Y=D\theta_0+g_0(X)+U,\qquad D=m_0(X)+V.$$

Oracle Mean DML 使用真实条件均值 $m_0(X)$ 与 $E[Y\mid X]$；Oracle Median DML 使用真实条件中位数。Oracle 已知真实函数，因此不训练梯度提升模型，也不需要 cross-fitting。

In [ ]:
# ============================================================
# 调参区
# ============================================================
P = 5
N = 1000
THETA_0 = 1.0
T_DF_GRID = [2.1, 2.3, 2.6, 3, 5, 10, 20]
LOGNORMAL_SIGMA_GRID = [2.0, 1.5, 1.25, 1.0, 0.75, 0.5, 0.25]
NORMAL_BASELINE = "N(0,1)"
DGP_NAMES = ["DGP1_linear", "DGP2_nonlinear"]

N_FOLDS = 5  # 仅记录与可行 DML 一致；Oracle 不需要交叉拟合
N_REPLICATIONS = 1000
BASE_SEED = 20260826
N_JOBS = -1

# 自己正式运行时保持 False；快速检查时改为 True
QUICK_TEST = False
QUICK_N = 300
QUICK_REPLICATIONS = 2

SAVE_RESULTS = True
OUTPUT_CSV = None  # 导入依赖后自动设为桌面 CSV

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from scipy.stats import norm

warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_theme(style="whitegrid", context="notebook")

ERROR_CASES = (
    [("Student-t", df, f"t({df:g})", order) for order, df in enumerate(T_DF_GRID)]
    + [("Student-t", NORMAL_BASELINE, NORMAL_BASELINE, len(T_DF_GRID))]
    + [("Lognormal", sigma, f"Lognormal(sigma={sigma:g})", order)
       for order, sigma in enumerate(LOGNORMAL_SIGMA_GRID)]
    + [("Lognormal", NORMAL_BASELINE, NORMAL_BASELINE, len(LOGNORMAL_SIGMA_GRID))]
)

if OUTPUT_CSV is None:
    OUTPUT_CSV = Path.home() / "Desktop" / "dml_oracle_p5_student_t_lognormal_summary.csv"
else:
    OUTPUT_CSV = Path(OUTPUT_CSV)

assert P == 5
assert all(df > 2 for df in T_DF_GRID)
assert all(sigma > 0 for sigma in LOGNORMAL_SIGMA_GRID)
print(f"参数检查完成；正式结果将保存到：{OUTPUT_CSV}")

## 1. 两类误差及 lognormal 中位数修正

Student-t 使用 $U=T_\nu\sqrt{(\nu-2)/\nu}$，因此均值、中位数均为 0，方差为 1。

Lognormal 使用
$$U=\frac{\exp(\sigma Z)-\exp(\sigma^2/2)}{\sqrt{(\exp(\sigma^2)-1)\exp(\sigma^2)}},\qquad Z\sim N(0,1),$$
因此均值为 0、方差为 1，但中位数不为 0。其真实中位数为
$$q_{0.5}(U)=\frac{1-\exp(\sigma^2/2)}{\sqrt{(\exp(\sigma^2)-1)\exp(\sigma^2)}}.$$
Oracle Median DML 必须从 $Y$ 中减去这个中位数修正；否则并不是真正的 Oracle 条件中位数。

In [ ]:
def nuisance_functions(X: np.ndarray, dgp_name: str):
    x1, x2, x3, x4, x5 = X.T
    if dgp_name == "DGP1_linear":
        m0 = 0.80*x1 - 0.50*x2 + 0.30*x3 + 0.20*x4 - 0.10*x5
        g0 = 1.00*x1 + 0.50*x2 - 0.50*x3 + 0.25*x4 + 0.10*x5
    elif dgp_name == "DGP2_nonlinear":
        m0 = (0.60*np.sin(x1) + 0.35*(x2**2 - 1.0)
              + 0.25*x3*x4 - 0.20*np.tanh(x5))
        g0 = (1.20*np.sin(x1) + 0.70*(x2**2 - 1.0)
              + 0.50*x3*x4 + 0.30*np.tanh(x5))
    else:
        raise ValueError(f"未知 DGP: {dgp_name}")
    return m0, g0


def standardized_student_t(rng, df: float, size: int):
    return rng.standard_t(df, size=size) * np.sqrt((df - 2.0) / df)


def standardized_centered_lognormal(rng, sigma: float, size: int):
    w = np.exp(sigma * rng.normal(0.0, 1.0, size=size))
    mean_w = np.exp(0.5*sigma**2)
    sd_w = np.sqrt(np.expm1(sigma**2) * np.exp(sigma**2))
    return (w - mean_w) / sd_w


def lognormal_error_median(sigma: float):
    mean_w = np.exp(0.5*sigma**2)
    sd_w = np.sqrt(np.expm1(sigma**2) * np.exp(sigma**2))
    return (1.0 - mean_w) / sd_w


def draw_error(rng, family: str, parameter, size: int):
    if parameter == NORMAL_BASELINE:
        return rng.normal(0.0, 1.0, size=size), 0.0
    if family == "Student-t":
        return standardized_student_t(rng, float(parameter), size), 0.0
    if family == "Lognormal":
        sigma = float(parameter)
        return standardized_centered_lognormal(rng, sigma, size), lognormal_error_median(sigma)
    raise ValueError(f"未知误差族：{family}")


# 完整 DGP 检查：真实条件均值和条件中位数的残差公式
rng_demo = np.random.default_rng(BASE_SEED)
X_demo = rng_demo.multivariate_normal(np.zeros(P), 0.5 ** np.abs(
    np.arange(P)[:, None] - np.arange(P)[None, :]
), size=300)
m_demo, g_demo = nuisance_functions(X_demo, "DGP2_nonlinear")
V_demo = rng_demo.normal(size=300)
U_demo, median_u_demo = draw_error(rng_demo, "Lognormal", 1.0, 300)
D_demo = m_demo + V_demo
Y_demo = THETA_0*D_demo + g_demo + U_demo
ell_mean_demo = THETA_0*m_demo + g_demo
ell_median_demo = ell_mean_demo + median_u_demo
np.testing.assert_allclose(D_demo - m_demo, V_demo, atol=1e-12)
np.testing.assert_allclose(Y_demo - ell_mean_demo, THETA_0*V_demo + U_demo, atol=1e-12)
np.testing.assert_allclose(Y_demo - ell_median_demo, THETA_0*V_demo + U_demo - median_u_demo, atol=1e-12)
print("完整 DGP 与 Oracle 均值/中位数残差检查通过。")

In [ ]:
def weighted_median(values: np.ndarray, weights: np.ndarray):
    order = np.argsort(values)
    values_sorted = values[order]
    weights_sorted = weights[order]
    cutoff = 0.5*weights_sorted.sum()
    index = np.searchsorted(np.cumsum(weights_sorted), cutoff, side="left")
    return float(values_sorted[min(index, len(values_sorted)-1)])


def fit_mean_oracle(V: np.ndarray, U: np.ndarray):
    d_res = V
    y_res = THETA_0*V + U
    jacobian = np.mean(d_res**2)
    theta_hat = np.mean(d_res*y_res) / jacobian
    score = d_res * (y_res - theta_hat*d_res)
    influence = score / jacobian
    se_hat = np.sqrt(np.var(influence, ddof=1) / len(d_res))
    return float(theta_hat), float(se_hat)


def fit_median_oracle(V: np.ndarray, U: np.ndarray, median_u: float):
    d_res = V
    y_res = THETA_0*V + U - median_u
    keep = np.abs(d_res) > 1e-10
    theta_hat = weighted_median(y_res[keep] / d_res[keep], np.abs(d_res[keep]))
    residual = y_res - theta_hat*d_res
    n = len(residual)
    mad = np.median(np.abs(residual - np.median(residual))) / 0.67448975
    robust_scale = max(float(mad), 1e-6)
    bandwidth = max(1.06*robust_scale*n**(-1.0/5.0), 1e-6)
    kernel_at_zero = norm.pdf(residual / bandwidth) / bandwidth
    A_hat = np.mean((d_res**2) * kernel_at_zero)
    B_hat = 0.25*np.mean(d_res**2)
    se_hat = np.sqrt(B_hat / (A_hat**2*n)) if A_hat > 1e-12 else np.nan
    return float(theta_hat), float(se_hat)

In [ ]:
def one_replication(case_id: int, replication: int, n_use: int):
    family, parameter, label, family_order = ERROR_CASES[case_id]
    seed = BASE_SEED + 10_000*case_id + replication
    rng = np.random.default_rng(seed)
    V = rng.normal(0.0, 1.0, size=n_use)
    U, median_u = draw_error(rng, family, parameter, n_use)
    estimates = [
        ("Oracle_Mean_DML", *fit_mean_oracle(V, U)),
        ("Oracle_Median_DML", *fit_median_oracle(V, U, median_u)),
    ]
    rows = []
    for dgp_name in DGP_NAMES:
        for estimator, theta_hat, se_hat in estimates:
            ci_low = theta_hat - 1.96*se_hat
            ci_high = theta_hat + 1.96*se_hat
            rows.append({
                "dgp": dgp_name, "p": P, "n": n_use,
                "error_family": family, "error_distribution": label,
                "family_order": family_order,
                "t_df": float(parameter) if family == "Student-t" and parameter != NORMAL_BASELINE else np.nan,
                "lognormal_sigma": float(parameter) if family == "Lognormal" and parameter != NORMAL_BASELINE else np.nan,
                "error_median": median_u, "replication": replication,
                "estimator": estimator, "theta_hat": theta_hat, "se_hat": se_hat,
                "ci_low": ci_low, "ci_high": ci_high,
                "covered": bool(ci_low <= THETA_0 <= ci_high),
                "sample_u_variance": np.var(U, ddof=1),
                "sample_max_abs_u": np.max(np.abs(U)),
            })
    return rows


def run_simulation():
    n_use = QUICK_N if QUICK_TEST else N
    r_use = QUICK_REPLICATIONS if QUICK_TEST else N_REPLICATIONS
    tasks = [(case_id, rep, n_use) for case_id in range(len(ERROR_CASES)) for rep in range(r_use)]
    mode = "QUICK_TEST" if QUICK_TEST else "FORMAL"
    print(f"开始 [{mode}]：p={P}, n={n_use}, error cases={len(ERROR_CASES)}, repetitions={r_use}")
    start = time.perf_counter()
    blocks = Parallel(n_jobs=N_JOBS, verbose=5)(delayed(one_replication)(*task) for task in tasks)
    rows = [row for block in blocks for row in block]
    print(f"完成，用时 {time.perf_counter()-start:.1f} 秒，共 {len(rows)} 条结果。")
    return pd.DataFrame(rows)


results = run_simulation()
results.head()

In [ ]:
def summarize_simulation(results_df: pd.DataFrame):
    work = results_df.copy()
    work["error"] = work["theta_hat"] - THETA_0
    work["squared_error"] = work["error"]**2
    summary = (work.groupby(
        ["dgp", "p", "error_family", "error_distribution", "family_order", "estimator"],
        as_index=False,
    ).agg(
        t_df=("t_df", "first"), lognormal_sigma=("lognormal_sigma", "first"),
        error_median=("error_median", "first"), mean_estimate=("theta_hat", "mean"),
        bias=("error", "mean"), sd=("theta_hat", "std"),
        mean_se=("se_hat", "mean"), mse=("squared_error", "mean"),
        coverage=("covered", "mean"),
        mean_sample_u_variance=("sample_u_variance", "mean"),
        mean_sample_max_abs_u=("sample_max_abs_u", "mean"),
        valid_runs=("theta_hat", "count"),
    ))
    summary["rmse"] = np.sqrt(summary.pop("mse"))
    return summary.sort_values(["error_family", "dgp", "family_order", "estimator"]).reset_index(drop=True)


summary = summarize_simulation(results)
display(summary.round({
    "mean_estimate": 4, "bias": 4, "sd": 4, "mean_se": 4,
    "rmse": 4, "coverage": 3, "error_median": 4,
    "mean_sample_u_variance": 3, "mean_sample_max_abs_u": 2,
}))

if SAVE_RESULTS and not QUICK_TEST:
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    summary.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"正式汇总结果已保存：{OUTPUT_CSV.resolve()}")
elif QUICK_TEST:
    print("QUICK_TEST=True：不写出测试 CSV。")
else:
    print("SAVE_RESULTS=False：未写出 CSV。")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for row, family in enumerate(["Student-t", "Lognormal"]):
    plot_data = summary[(summary["error_family"] == family) & (summary["dgp"] == "DGP1_linear")]
    for ax, (metric, label) in zip(axes[row], [("bias", "Bias"), ("rmse", "RMSE"), ("coverage", "95% CI coverage")]):
        sns.lineplot(data=plot_data, x="family_order", y=metric, hue="estimator", markers=True, ax=ax)
        labels = (
            [f"t({x:g})" for x in T_DF_GRID] + [NORMAL_BASELINE]
            if family == "Student-t"
            else [f"LN({x:g})" for x in LOGNORMAL_SIGMA_GRID] + [NORMAL_BASELINE]
        )
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=30)
        ax.set_title(f"{family}: {label}")
        ax.set_xlabel("Error distribution")
        ax.set_ylabel(label)
        if metric == "bias": ax.axhline(0.0, color="black", linewidth=1, linestyle=":")
        if metric == "coverage":
            ax.axhline(0.95, color="black", linewidth=1, linestyle=":")
            ax.set_ylim(0.0, 1.02)
plt.tight_layout()
plt.show()

## 使用说明

1. 自己正式运行时保持 `QUICK_TEST=False`，从头运行全部单元。
2. 正式运行完成后，汇总结果自动保存到桌面的 `dml_oracle_p5_student_t_lognormal_summary.csv`。
3. Gaussian baseline 在两个误差族下各保留一次，便于分别与 Student-t 和 lognormal 网格比较。
4. 两个 DGP 的 Oracle 汇总结果理论上相同，因为真实 nuisance functions 被精确消去；保留 DGP 标签是为了与原结果直接合并比较。
5. 若 Gaussian Oracle coverage 接近 0.95，而原 DML 明显偏低，则说明主要问题来自第一阶段 nuisance estimation。